# Integração SINAN–IBGE/SIDRA

Este notebook reproduz somente a parte necessária do tratamento do SINAN; o arquivo `sinan.ipynb` permanece inalterado. Primeiro são analisadas qualidade e distribuição das notificações, depois são realizados os merges municipais nos anos compatíveis.

In [1]:
import polars as pl
import plotly.express as px

from analysis_pipeline import build_integrated_outputs

pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_cols(14)

polars.config.Config

In [2]:
dados = build_integrated_outputs(refresh_ibge=False)
painel = dados['populacao_integrada']
integrado_2010 = dados['saneamento_integrado']
integrado_2022 = dados['saneamento_2022_integrado']
dados['auditoria']

base,linhas_totais,linhas_elegiveis,correspondencias,percentual_correspondencia
str,i64,i64,i64,f64
"""populacao""",95272,80482,80473,99.988817
"""saneamento_2010""",4307,4307,4307,100.0
"""saneamento_2022""",4678,4678,4678,100.0


## 1. Auditoria do SINAN

Os 24 arquivos anuais possuem esquemas diferentes. O pipeline harmoniza `DENGUE` com `CLASSI_FIN`, `CON_EVOLUC` com `EVOLUCAO`, datas em dois formatos e códigos municipais de seis ou sete dígitos. O município utilizado é o de notificação (`ID_MUNICIP`).

In [3]:
qualidade_total = dados['qualidade'].select(
    pl.col('registros').sum(),
    pl.col('codigo_municipio_invalido').sum(),
    pl.col('data_invalida').sum(),
    pl.col('ano_informado_divergente').sum(),
    pl.col('ano_data_divergente').sum(),
)
qualidade_total

registros,codigo_municipio_invalido,data_invalida,ano_informado_divergente,ano_data_divergente
u32,u32,u32,u32,u32
25031001,55,1,24202,23688


### Definição do desfecho

Casos confirmados incluem `DENGUE=1` na ficha antiga e os códigos históricos/atuais confirmatórios `1–4` e `10–12` em `CLASSI_FIN`. Descartados, inconclusivos, ignorados e não preenchidos permanecem separados.

In [4]:
classificacao = (
    dados['classificacao']
    .group_by('status_classificacao')
    .agg(pl.col('registros').sum())
    .with_columns(
        (pl.col('registros') / pl.col('registros').sum() * 100).alias('percentual')
    )
    .sort('registros', descending=True)
)
display(classificacao)
px.bar(
    classificacao.to_pandas(), x='status_classificacao', y='registros',
    text_auto='.3s', title='Classificação das notificações SINAN — 2000 a 2023',
)

status_classificacao,registros,percentual
str,u32,f64
"""confirmado""",12999322,51.932889
"""descartado""",8218128,32.831799
"""inconclusivo""",2897602,11.576053
"""nao_preenchido""",634516,2.534921
"""ignorado""",272755,1.089669
"""outro_codigo""",8678,0.034669


## 2. Evolução temporal antes do merge

In [5]:
serie_anual = (
    painel
    .group_by('ano')
    .agg(
        pl.col('notificacoes').sum(),
        pl.col('casos_confirmados').sum(),
        pl.col('codigo_municipio').n_unique().alias('municipios'),
    )
    .sort('ano')
)
display(serie_anual)
px.line(
    serie_anual.to_pandas(), x='ano', y=['notificacoes', 'casos_confirmados'],
    markers=True, title='Notificações e casos confirmados por ano',
)

ano,notificacoes,casos_confirmados,municipios
i32,u32,u32,u32
2000,172855,10758,1788
2001,488590,28681,3233
2002,897093,71283,4006
2003,416609,41727,3554
2004,136867,15816,2800
2005,261500,26076,3167
2006,411022,34618,3556
2007,717097,346292,4080
…,…,…,…


In [6]:
serie_mensal = dados['mensal'].to_pandas()
px.line(
    serie_mensal, x='mes', y='notificacoes', color='ano',
    title='Sazonalidade mensal das notificações por ano',
).update_xaxes(dtick=1)

## 3. Completude de etnia

In [7]:
def totais_categoricos(df: pl.DataFrame, coluna: str) -> pl.DataFrame:
    return (
        df.group_by(coluna)
        .agg(pl.col('registros').sum())
        .with_columns(
            (pl.col('registros') / pl.col('registros').sum() * 100).alias('percentual')
        )
        .sort('registros', descending=True)
    )

display(totais_categoricos(dados['raca'], 'raca'))
display(totais_categoricos(dados['evolucao'], 'evolucao'))

raca,registros,percentual
str,u32,f64
"""branca""",8109925,32.399523
"""parda""",7629891,30.481765
"""ignorado""",5062069,20.223198
"""nao_preenchido""",3001801,11.992333
"""preta""",958584,3.829587
"""amarela""",206164,0.823635
"""indigena""",62567,0.249958


evolucao,registros,percentual
str,u32,f64
"""cura""",18551731,74.115018
"""nao_preenchido""",5363513,21.427481
"""ignorado""",776927,3.103859
"""nao_se_aplica""",306096,1.222868
"""obito_outras_causas""",17613,0.070365
"""obito_pelo_agravo""",13360,0.053374
"""obito_em_investigacao""",1761,0.007035


## 4. Integração municipal

A tabela 3218 é associada somente às notificações de 2010. O bloco Censo 2022 (6803, 6805 e 6892) é associado somente a 2022. A população vem das estimativas 6579 e, em 2022, do Censo 4709. Nenhum valor de saneamento é replicado para outro ano.

In [8]:
resumo_recortes = pl.DataFrame({
    'ano': [2010, 2022],
    'municipios_sinan': [integrado_2010.height, integrado_2022.height],
    'municipios_com_sidra': [
        integrado_2010['correspondencia_sidra'].sum(),
        integrado_2022['correspondencia_sidra'].sum(),
    ],
    'notificacoes': [
        integrado_2010['notificacoes'].sum(),
        integrado_2022['notificacoes'].sum(),
    ],
    'casos_confirmados': [
        integrado_2010['casos_confirmados'].sum(),
        integrado_2022['casos_confirmados'].sum(),
    ],
})
resumo_recortes

ano,municipios_sinan,municipios_com_sidra,notificacoes,casos_confirmados
i64,i64,i64,i64,i64
2010,4307,4307,1381254,869248
2022,4678,4678,1405095,1254238


## 5. Associação exploratória em 2022

In [9]:
indicadores_2022 = [
    'pct_agua_rede_geral_principal',
    'pct_esgoto_rede_geral_ou_pluvial',
    'pct_sem_banheiro_ou_sanitario',
    'pct_lixo_queimado_propriedade',
    'pct_lixo_terreno_encosta_area_publica',
]
correlacoes = pl.DataFrame({
    'indicador': indicadores_2022,
    'correlacao_pearson_taxa_confirmados': [
        integrado_2022.select(pl.corr(coluna, 'taxa_confirmados_100k')).item()
        for coluna in indicadores_2022
    ],
}).sort('correlacao_pearson_taxa_confirmados', descending=True)
correlacoes

indicador,correlacao_pearson_taxa_confirmados
str,f64
"""pct_agua_rede_geral_principal""",0.146407
"""pct_esgoto_rede_geral_ou_pluvi…",0.100838
"""pct_lixo_terreno_encosta_area_…",-0.111232
"""pct_sem_banheiro_ou_sanitario""",-0.113783
"""pct_lixo_queimado_propriedade""",-0.151607


In [10]:
px.scatter(
    integrado_2022.to_pandas(),
    x='pct_esgoto_rede_geral_ou_pluvial',
    y='taxa_confirmados_100k',
    hover_name='municipio',
    opacity=0.45,
    title='Esgotamento e taxa de casos confirmados — municípios, 2022',
)

## Interpretação e limites

- A unidade de análise é município–ano, não pessoa.
- O município é o de notificação, que pode diferir do município de residência.
- As relações são descritivas e suscetíveis a confundimento, subnotificação, diferenças na vigilância e falácia ecológica.
- Correlação não implica causalidade; modelagem futura deve incluir clima, estrutura demográfica, efeitos espaciais e validação temporal.
- Os contratos finais são gravados em `sinan/processados/`.